# 5차시 (1) 프롬프트 엔지니어링과 나만의 챗봇 — 실습

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

이 노트북에서 우리는 **두 가지**를 직접 해 봅니다.

1. **프롬프트 엔지니어링** — 같은 AI 모델이라도 *어떻게 물어보느냐* 에 따라 답이 크게 달라집니다. 좋은 질문(프롬프트)을 쓰는 4가지 방법을 실습합니다.
2. **나만의 Gradio 챗봇** — 코드 몇 줄로 웹 채팅 화면을 띄우고, **시스템 프롬프트(페르소나)** 를 바꿔 가며 *성격이 다른 챗봇* 을 만들어 봅니다.

앞 차시까지 파이썬으로 데이터를 다루고 머신러닝·딥러닝 모델을 **가져다 써** 봤다면, 이제 그 파이썬 실력으로 LLM을 직접 호출해 **나만의 챗봇** 을 만들어 봅니다.

> 처음 보는 코드가 많아도 괜찮습니다. **셀을 위에서부터 차례대로 ▶ 실행**하면서 따라오면 됩니다. 중간에 `# TODO` 가 보이면 직접 채워 보는 부분입니다.

## 준비물 한눈에 보기

- **실행 환경**: Google Colab (이 노트북). 설치할 것은 아래 셀이 알아서 받아 줍니다.
- **OpenAI API 키**: 강의 주최측에서 발급해 드립니다. 키는 **이 노트북 안에서 입력**하며, 코드에 직접 적지 않습니다(보안).
- **이번 노트북에서 쓰는 AI 모델**: `gpt-4o-mini` — 저렴하고 빠른 모델이라 실습에 적합합니다.

순서: ① 라이브러리 설치 → ② API 키 입력 → ③ Part 1 프롬프트 실습 → ④ Part 2 챗봇 만들기.

---
## 0. 환경 세팅

### 0-1. 라이브러리 설치
LangChain(언어 모델을 편하게 부르는 도구)과 Gradio(웹 챗봇 화면 도구)를 설치합니다. **30초~1분** 정도 걸립니다.
설치 후 `pip` 의존성 관련 빨간 경고가 한두 줄 떠도 **무시해도 됩니다**(실행에는 지장 없음).

In [ ]:
# -qU : 조용히(-q) + 최신 버전으로 업그레이드(-U) 설치
!pip install -qU langchain-openai langchain-core
!pip install -qU "gradio>=5.0"
print("설치 완료! 위에 경고가 보여도 괜찮습니다.")

### 0-2. OpenAI API 키 입력
아래 셀을 실행하면 입력창이 나옵니다. **주최측에서 받은 키를 붙여넣고 Enter** 를 누르세요.
- `getpass` 를 쓰기 때문에 입력한 키는 **화면에 보이지 않습니다**(어깨너머로 노출되지 않게).
- 키는 `OPENAI_API_KEY` 라는 환경변수에 저장되어, 아래 모든 셀이 자동으로 사용합니다.
- 런타임을 다시 시작(연결 끊김)하면 이 셀을 **다시 실행**해야 합니다.

In [ ]:
import os
import getpass

# 이미 입력했다면 다시 묻지 않습니다.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키를 입력하세요: ")

print("API 키가 설정되었습니다." if os.environ.get("OPENAI_API_KEY") else "키가 비어 있습니다. 다시 실행하세요.")

### 0-3. 모델 한 번 불러보기 (연결 확인)
키가 잘 설정됐는지, 모델이 실제로 답하는지 **가장 간단한 한 줄**로 확인합니다.

- `ChatOpenAI` : OpenAI의 채팅 모델을 부르는 객체.
- `temperature` : 답변의 *무작위성/창의성*. 0에 가까울수록 매번 비슷하고 딱딱하게, 1에 가까울수록 다양하고 자유롭게 답합니다.
- `.invoke("...")` : 모델에게 메시지를 보내고 답을 받아오는 함수.

In [ ]:
from langchain_openai import ChatOpenAI

# 이번 실습 내내 쓸 모델. 저렴하고 빠른 gpt-4o-mini 사용.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

answer = llm.invoke("한 문장으로 자기소개를 해줘.")
print(answer.content)

---
# Part 1 · 프롬프트 엔지니어링

**프롬프트(prompt)** = 우리가 AI에게 건네는 *지시문/질문* 입니다.
모델은 그대로 두고 **프롬프트만 잘 바꿔도** 답의 품질이 확 올라갑니다. 이것을 다듬는 일이 **프롬프트 엔지니어링** 입니다.

아래 4가지 방법을 *나쁜 예 → 좋은 예* 로 직접 비교해 봅니다.

1. **명료하게 지시하기** — 구체적으로, 빠짐없이 요청
2. **참고 텍스트 제공하기** — 답의 근거가 될 자료를 함께 줌
3. **작업을 작게 쪼개기** — 큰 일을 단계로 나눠 시킴
4. **페르소나(역할) 부여** — "너는 ○○야" 로 말투·전문성 지정

> 참고: OpenAI 공식 프롬프트 가이드 — https://developers.openai.com/api/docs/guides/prompt-engineering

먼저, 비교를 편하게 하려고 **프롬프트를 넣으면 답을 출력하는 작은 함수** 를 하나 만들어 둡니다.

In [ ]:
def ask(prompt, temperature=0.7):
    """프롬프트(문자열)를 모델에 보내고 답변 텍스트를 반환 + 화면에 출력."""
    model = ChatOpenAI(model="gpt-4o-mini", temperature=temperature)
    result = model.invoke(prompt)
    print(result.content)
    return result.content

## 1-1. 명료하게 지시하기
모호한 질문은 모호한 답을 부릅니다. **무엇을, 어떤 형식으로, 어떤 조건으로** 원하는지 구체적으로 적으면 답이 좋아집니다.

아래 두 셀을 각각 실행해 **답의 차이**를 비교해 보세요.

In [ ]:
# (나쁜 예) 너무 막연한 질문
print("===== 모호한 프롬프트 =====")
ask("피보나치 수열 코드 짜줘.")

In [ ]:
# (좋은 예) 언어, 조건, 형식, 설명까지 구체적으로
print("===== 구체적인 프롬프트 =====")
ask(
    "파이썬으로 피보나치 수열의 n번째 값을 구하는 함수를 작성해줘. "
    "조건: ① 반복문 사용(재귀 금지), ② 입력이 음수면 안내 메시지 출력, "
    "③ 각 줄에 무엇을 하는지 한국어 주석을 달아줘."
)

**TODO 1.** 아래 빈칸에 *여러분이 궁금한 주제* 로 "모호한 질문 → 구체적인 질문" 한 쌍을 만들어 실행해 보세요.
(예: 여행 코스 추천, 이메일 작성, 요약 등)

In [ ]:
# TODO: 모호하게 한 번, 구체적으로 한 번 물어보고 답을 비교해 보세요.
ask("여행 추천해줘.")                 # 모호한 예 (자유롭게 바꿔도 됨)
print("-" * 40)
ask("")  # TODO: 여기에 '대상/일정/예산/관심사'를 담아 구체적으로 작성

## 1-2. 참고할 텍스트를 함께 제공하기
모델은 *모르는 사실*(최신 정보, 우리 회사 내부 자료 등)은 지어낼 수 있습니다(이를 **환각, hallucination** 이라 합니다).
이때 **답의 근거가 될 텍스트를 프롬프트에 같이 넣어** 주고 "이 내용만 보고 답하라"고 지시하면 훨씬 정확해집니다.

> 이 아이디어를 문서 전체로 확장한 것이 바로 **다음 노트북의 RAG** 입니다.

In [ ]:
# 모델이 학습 시점에 몰랐을 법한 '가상의 제품 설명'을 직접 제공
reference = """
[제품 안내문 — 한경 스마트텀블러 X2]
- 용량: 500ml
- 보온: 12시간 / 보냉: 24시간
- 무게: 320g
- 색상: 미드나잇 블랙, 아이보리, 포레스트 그린
- 가격: 32,000원
- 보증: 구매일로부터 1년
"""

question = "이 텀블러 보냉은 몇 시간이고 가격은 얼마야? 안내문에 없는 내용은 '안내문에 없음'이라고 답해줘."

prompt = f"""아래 안내문만 참고해서 질문에 답해줘.

안내문:
{reference}

질문: {question}"""

ask(prompt, temperature=0)   # 사실 질문이므로 temperature=0(딱딱하지만 일관됨)

**TODO 2.** 위 `reference` 안내문에 **없는** 내용(예: "방수가 되나요?")을 물어보세요.
근거가 없을 때 모델이 *"안내문에 없음"* 이라고 답하는지 확인해 보세요 — 이것이 환각을 줄이는 핵심입니다.

In [ ]:
# TODO: 안내문에 없는 질문을 넣어 '안내문에 없음'이라고 답하는지 확인
new_question = ""  # 예: "이 텀블러는 식기세척기를 써도 되나요?"
prompt = f"""아래 안내문만 참고해서 질문에 답해줘. 없는 내용은 '안내문에 없음'이라고 답해.

안내문:
{reference}

질문: {new_question}"""
ask(prompt, temperature=0)

## 1-3. 복잡한 작업은 작은 단계로 쪼개기
큰 요청을 한 번에 던지면 빠뜨리는 부분이 생깁니다. **순서가 있는 단계(1·2·3…)로 나눠** 지시하면 결과가 더 완성도 높고 일관됩니다.

아래는 "공 튀기기 게임 만들기"를 *한 줄 요청* vs *단계별 요청* 으로 비교합니다.
(게임 코드를 Colab에서 실행하진 않습니다 — **생성된 코드의 완성도 차이**만 비교하세요.)

In [ ]:
# (나쁜 예) 한 줄 요청
print("===== 한 줄 요청 =====")
ask("파이썬으로 공 튀기기 게임 코드 만들어줘.")

In [ ]:
# (좋은 예) 단계로 쪼갠 요청
print("===== 단계별 요청 =====")
ask(
    "파이썬 pygame으로 '공 튀기기 게임'을 아래 단계 순서대로 빠짐없이 구현해줘.\n"
    "1) 게임 창(800x600) 생성\n"
    "2) 원형 공 객체 정의(초기 위치·속도 포함)\n"
    "3) 공이 벽에 부딪히면 튕기게 만들기\n"
    "4) 게임 루프에서 매 프레임 위치 갱신·화면 다시 그리기\n"
    "5) 방향키로 공의 속도를 바꾸기\n"
    "6) 위 1~5를 합친 전체 코드를 한 번에 제시"
)

## 1-4. 페르소나(역할) 부여하기
"너는 ○○야" 라고 **역할을 정해** 주면 말투·난이도·전문성이 그 역할에 맞게 바뀝니다.
같은 질문을 **두 가지 페르소나**에게 던져 답이 어떻게 달라지는지 비교해 봅시다.

In [ ]:
question = "벡터(vector)가 뭔지 설명해줘."

print("===== 페르소나 A: 초등학생에게 설명하는 친절한 선생님 =====")
ask(f"너는 초등학생에게 설명하는 친절한 선생님이야. 쉬운 비유와 이모지를 써서 설명해줘.\n질문: {question}")

print("\n" + "=" * 50 + "\n")

print("===== 페르소나 B: 대학생에게 설명하는 데이터 사이언스 전문가 =====")
ask(f"너는 데이터 사이언스 전문가야. 대학생에게 수학적 정의를 포함해 전문적으로 설명해줘.\n질문: {question}")

**TODO 3.** 나만의 페르소나를 하나 정해서(예: *조선시대 선비*, *축구 해설가*, *깐깐한 코드 리뷰어*) 같은 질문에 답하게 해 보세요.

In [ ]:
# TODO: persona 문장과 my_question 을 자유롭게 바꿔 실행해 보세요.
persona = "너는 (역할)이야. (말투/특징)을 유지해서 답해줘."   # TODO: (역할)·(말투) 채우기
my_question = "좋아하는 계절을 추천해줘."                      # TODO: 자유 질문
ask(f"{persona}\n질문: {my_question}")

### Part 1 정리
- **명료한 지시 / 참고 텍스트 / 단계 분할 / 페르소나** — 모델은 그대로여도 프롬프트만으로 답이 크게 좋아집니다.
- 특히 **참고 텍스트 제공**은 다음 노트북에서 배울 **RAG** 의 출발점입니다.
- 이제 이 프롬프트들을 *매번 코드로 치지 않고* **챗봇 화면**에서 편하게 시험해 봅시다.

---
# Part 2 · 나만의 Gradio 챗봇 만들기

지금까지는 셀에서 한 번씩 질문했습니다. 이제 **웹 채팅 화면**을 띄워서, 사람처럼 *대화를 주고받는* 챗봇을 만들어 봅니다.

도구는 **Gradio** 입니다. 함수 하나만 정의하면 채팅 UI를 자동으로 만들어 줍니다.
핵심 개념 두 가지:
- **시스템 프롬프트(system prompt)**: 챗봇의 *성격·역할·규칙* 을 정하는 숨은 지시문. 사용자에게는 안 보이지만 모든 답변에 영향을 줍니다. = **페르소나의 핵심**.
- **대화 기록(history)**: 앞에서 무슨 말을 했는지 모델에게 같이 넘겨야 *맥락을 기억하는* 대화가 됩니다.

## 2-1. 가장 간단한 챗봇 (입력을 그대로 따라 말하기)
먼저 AI 없이 **Gradio 채팅창이 어떻게 동작하는지** 감을 잡습니다.
`gr.ChatInterface` 는 우리가 만든 `response` 함수를 호출해 답을 화면에 보여줍니다.

- `message` : 사용자가 방금 친 말
- `history` : 지금까지의 대화 기록 (`{"role": "user"/"assistant", "content": ...}` 형태의 목록)

> 실행하면 셀 아래에 채팅창이 뜹니다. Colab에서는 **공개 URL(`https://...gradio.live`)** 도 함께 생성됩니다(1주일간 유효).
> 다 써 본 뒤에는 **다음 챗봇을 켜기 전에** 바로 아래의 `demo.close()` 셀을 실행해 이전 창을 꺼 주세요(포트 충돌 방지).

In [ ]:
import gradio as gr

def echo_response(message, history):
    # AI를 부르지 않고, 받은 말을 그대로 따라 합니다.
    return f"당신이 말한 내용: {message}"

demo = gr.ChatInterface(
    fn=echo_response,
    type="messages",                       # 최신 Gradio 메시지 형식(필수)
    title="따라 말하기 챗봇",
    description="아직 AI가 아니라, 입력을 그대로 돌려주는 연습용 챗봇입니다.",
)
demo.launch()

In [ ]:
# 위 챗봇을 다 써 봤다면 이 셀을 실행해 창을 닫습니다(다음 챗봇 실행 전 권장).
demo.close()

## 2-2. 진짜 AI 챗봇 (시스템 프롬프트 + 대화 기억)
이제 `response` 함수 안에서 **실제 모델을 호출**합니다. 두 가지를 챙깁니다.

1. **시스템 프롬프트**를 맨 앞에 넣어 챗봇의 *성격* 을 지정합니다.
2. **`history` 를 모델에게 전달**해 이전 대화를 기억하게 합니다.

LangChain에서는 메시지를 역할별 객체로 만듭니다:
- `SystemMessage` (규칙/페르소나) · `HumanMessage` (사용자) · `AIMessage` (챗봇)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr

# 이 챗봇의 성격을 정하는 시스템 프롬프트
SYSTEM_PROMPT = "너는 친절하고 상냥한 AI 비서야. 사용자의 말에 공감하며 한국어로 답해줘."

chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def chat_response(message, history):
    # 1) 맨 앞에 시스템 프롬프트(성격)를 넣는다
    messages = [SystemMessage(content=SYSTEM_PROMPT)]

    # 2) 지금까지의 대화 기록을 역할에 맞는 메시지로 변환해 추가
    for turn in history:
        if turn["role"] == "user":
            messages.append(HumanMessage(content=turn["content"]))
        else:
            messages.append(AIMessage(content=turn["content"]))

    # 3) 사용자가 방금 친 말을 마지막에 추가
    messages.append(HumanMessage(content=message))

    # 4) 모델 호출 후 답변 텍스트만 반환
    return chat_llm.invoke(messages).content

demo = gr.ChatInterface(
    fn=chat_response,
    type="messages",
    title="나만의 AI 챗봇",
    description="시스템 프롬프트로 성격을 정한 챗봇과 대화해 보세요.",
    examples=["안녕! 오늘 기분이 좀 우울해.", "주말에 할 만한 취미 추천해줘.", "피보나치 수열을 쉽게 설명해줘."],
)
demo.launch()

In [ ]:
demo.close()

## 2-3. 성격 바꿔 보기 — 화면에서 시스템 프롬프트 수정
매번 코드를 고치지 않고, **채팅 화면에서 시스템 프롬프트를 직접 입력**해 성격을 바꿔 봅시다.
`additional_inputs` 로 입력칸을 하나 더 달면, 그 값이 `chat_response` 의 세 번째 인자로 들어옵니다.

채팅창 아래(또는 'Additional Inputs' 펼치기) 에서 시스템 프롬프트를 바꾸고 다시 말을 걸어 보세요. 예시:
- `너는 8살 아이야. 아이처럼 신나고 짧게 말해줘.`
- `너는 츤데레 고양이 집사야. 살짝 퉁명스럽지만 결국 도와줘.`
- `너는 깐깐한 면접관이야. 짧고 날카롭게 되묻는다.`

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr

chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)

def persona_response(message, history, system_prompt):
    # system_prompt 가 비어 있으면 기본값 사용
    sp = system_prompt.strip() or "너는 친절한 AI 비서야. 한국어로 답해줘."
    messages = [SystemMessage(content=sp)]
    for turn in history:
        role = HumanMessage if turn["role"] == "user" else AIMessage
        messages.append(role(content=turn["content"]))
    messages.append(HumanMessage(content=message))
    return chat_llm.invoke(messages).content

demo = gr.ChatInterface(
    fn=persona_response,
    type="messages",
    title="성격을 바꿀 수 있는 챗봇",
    description="아래 시스템 프롬프트를 바꾸면 챗봇의 성격이 달라집니다.",
    additional_inputs=[
        gr.Textbox(
            value="너는 8살 아이야. 아이처럼 신나고 짧게 말해줘.",
            label="시스템 프롬프트 (챗봇의 성격)",
            lines=3,
        )
    ],
)
demo.launch()

In [ ]:
demo.close()

## 2-4. (실습 과제) 나만의 자기소개서 챗봇 만들기
배운 것을 모아 **자기소개서 작성을 도와주는 챗봇**을 직접 완성해 보세요.

**미션**: 아래 `MY_SYSTEM_PROMPT` 를 채워, 사용자의 경험을 물어보고 자기소개서 문단을 써 주는 챗봇을 만드세요.
좋은 시스템 프롬프트에 담으면 좋은 것:
- **역할**: "너는 취업 자기소개서를 도와주는 컨설턴트야."
- **방식**: 한 번에 다 쓰지 말고 **필요한 정보를 먼저 질문** 하게 하기(작업 쪼개기!).
- **포함 항목**(사용자에게 물어볼 것): ① 인상 깊었던 전공 과목 ② 갈등을 해결한 경험 ③ 건강한 취미.
- **말투/분량**: 정중한 존댓말, 한 문단 5~7문장 등.

답이 마음에 안 들면 *"더 구체적으로", "겸손한 톤으로 다시"* 처럼 **수정 요청**도 해 보세요.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr

# TODO: 아래 따옴표 사이에 '자기소개서 도우미' 시스템 프롬프트를 직접 작성하세요.
MY_SYSTEM_PROMPT = """
(여기에 챗봇의 역할·질문 방식·포함 항목·말투를 적으세요)
"""

chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def resume_response(message, history):
    sp = MY_SYSTEM_PROMPT.strip() or "너는 자기소개서 작성을 도와주는 컨설턴트야. 한국어 존댓말로 답해줘."
    messages = [SystemMessage(content=sp)]
    for turn in history:
        role = HumanMessage if turn["role"] == "user" else AIMessage
        messages.append(role(content=turn["content"]))
    messages.append(HumanMessage(content=message))
    return chat_llm.invoke(messages).content

demo = gr.ChatInterface(
    fn=resume_response,
    type="messages",
    title="나만의 자기소개서 챗봇",
    description="경험을 입력하면 자기소개서 문단을 만들어 줍니다.",
    examples=["자기소개서 작성을 도와줘.", "내가 인상 깊게 들은 과목은 '머신러닝 입문'이야.", "방금 문단을 더 겸손한 톤으로 다시 써줘."],
)
demo.launch()

In [ ]:
demo.close()

---
## 마무리

이 노트북에서 한 것:
- **프롬프트 엔지니어링 4원칙**(명료한 지시 · 참고 텍스트 · 작업 분할 · 페르소나)을 직접 비교 실습.
- **Gradio 챗봇**을 만들고, **시스템 프롬프트(페르소나)** 와 **대화 기록** 으로 성격 있는 대화를 구현.

**다음 노트북 예고 — RAG 챗봇**: 1-2에서 *참고 텍스트를 직접 붙여넣어* 답하게 했던 것을, **여러 문서에서 자동으로 관련 부분을 찾아** 답하도록 확장합니다. "내 문서로 답하는 챗봇"을 만들어 봅시다.